# V18-v2 Result 4.3 CR Verification — IT-Perspective Trajectory

**Core question:** For each CR-specific finding, is it:
1. **IT-persistent** → IT에서 시작, IA 거쳐 CR까지 지속 (NL→IT sig, NL→CR sig)
2. **IA-persistent** → IA에서 새로 생겨서 CR까지 지속 (NL→IT NS, NL→IA sig, NL→CR sig)
3. **CR-unique** → IT/IA에 없고 CR에서만 출현 (NL→IT NS, NL→IA NS, NL→CR sig)

**Target genes:** Table 5 genes + additional CR-relevant markers
- From-zero: MEFV, COL1A1, FN1, NLRP3
- Quantitative: DNMT3A, GNLY (PlasmaB)
- NK depletion markers
- Plus: all six-layer genes to check IT→CR persistence

**Method:** 5 comparisons per gene-lineage-tissue:
NL→IT, NL→IA, NL→CR, IT→CR, IA→CR

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 142.2 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [2]:
# Cell 1: Setup
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import os, time, warnings
warnings.filterwarnings('ignore')

try:
    _ = adata.shape
    print(f'adata loaded: {adata.shape}')
except:
    from google.colab import drive
    drive.mount('/content/drive')
    import scanpy as sc
    DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
    print('Loading h5ad...')
    adata = sc.read_h5ad(DATA_PATH)
    print(f'Loaded: {adata.shape}')

RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'
SAVE_DIR = os.path.join(RESULTS_DIR, 'Result4-3-supplementary_26Mar13')
os.makedirs(SAVE_DIR, exist_ok=True)

obs = adata.obs.copy()

# Column detection
COL_STAGE = 'Stage' if 'Stage' in obs.columns else [c for c in obs.columns if 'stage' in c.lower()][0]
COL_LINEAGE = 'major_lineage' if 'major_lineage' in obs.columns else [c for c in obs.columns if 'lineage' in c.lower()][0]
if 'tissue' in obs.columns:
    COL_TISSUE = 'tissue'
elif 'Tissue' in obs.columns:
    COL_TISSUE = 'Tissue'
else:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['tissue_derived'] = obs[c].apply(
                lambda x: 'Liver' if ('Liver' in str(x) or '_L_' in str(x) or str(x).endswith('_L'))
                else ('Blood' if ('PBMC' in str(x) or '_P_' in str(x) or str(x).endswith('_P') or 'Blood' in str(x))
                else 'Unknown'))
            COL_TISSUE = 'tissue_derived'
            break
COL_DONOR = None
for c in ['donor', 'Donor', 'patient', 'subject', 'donor_id']:
    if c in obs.columns:
        COL_DONOR = c
        break
if COL_DONOR is None:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['donor_derived'] = obs[c].astype(str).str.split('_').str[1]
            COL_DONOR = 'donor_derived'
            break

print(f'Columns: Stage={COL_STAGE}, Lineage={COL_LINEAGE}, Tissue={COL_TISSUE}, Donor={COL_DONOR}')

Mounted at /content/drive
Loading h5ad...
Loaded: (243000, 24452)
Columns: Stage=Stage, Lineage=major_lineage, Tissue=tissue, Donor=donor_derived


In [3]:
# Cell 2: Pre-extract genes
t0 = time.time()

target_genes = [
    # Table 5 CR scar genes
    'MEFV', 'COL1A1', 'FN1', 'NLRP3',
    # Table 5 quantitative CR
    'DNMT3A', 'GNLY',
    # Six-layer genes — check IT→CR persistence
    'TGFB1', 'LGALS9', 'DNMT1', 'TET2', 'MTOR', 'LDHA',
    'SOCS1', 'SOCS3', 'TOX', 'LAYN', 'PRDM1', 'RORC',
    'JAK1', 'AIM2', 'FOXP3', 'TIGIT', 'BCL6',
    # Cytotoxic markers
    'GZMB', 'GZMK', 'PRF1',
    # Additional
    'TYROBP', 'FCER1G', 'BAK1', 'HLA-DPB1',
]
target_genes = [g for g in target_genes if g in adata.var_names]
print(f'Genes to test: {len(target_genes)}')

gene_expr = {}
for gi, gene in enumerate(target_genes):
    try:
        col = adata[:, gene].X
        if hasattr(col, 'toarray'):
            col = col.toarray().flatten()
        else:
            col = np.asarray(col).flatten()
        gene_expr[gene] = col
    except Exception as e:
        print(f'  ⚠️ {gene}: {e}')

print(f'Pre-extraction done in {time.time()-t0:.1f}s ({len(gene_expr)} genes)')

Genes to test: 30
Pre-extraction done in 9.0s (30 genes)


In [4]:
# Cell 3: Fast donor-level test function

def fast_donor_test(gene, lineage, tissue, group_a, group_b):
    if gene not in gene_expr:
        return None
    mask = (
        (obs[COL_STAGE].isin([group_a, group_b])) &
        (obs[COL_LINEAGE] == lineage) &
        (obs[COL_TISSUE] == tissue)
    )
    if mask.sum() == 0:
        return None
    cell_indices = np.where(mask.values)[0]
    expr = gene_expr[gene][cell_indices]
    temp = obs.loc[mask, [COL_DONOR, COL_STAGE]].copy()
    temp['expr'] = expr
    donor_means = temp.groupby([COL_DONOR, COL_STAGE])['expr'].mean().reset_index()
    vals_a = donor_means[donor_means[COL_STAGE] == group_a]['expr'].dropna().values
    vals_b = donor_means[donor_means[COL_STAGE] == group_b]['expr'].dropna().values
    n_a, n_b = len(vals_a), len(vals_b)
    if n_a < 2 or n_b < 2:
        return None
    mean_a, mean_b = float(np.mean(vals_a)), float(np.mean(vals_b))
    try:
        _, p_val = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        p_val = float(p_val)
    except:
        p_val = 1.0
    if mean_a > 1e-10:
        pct = (mean_b - mean_a) / mean_a * 100
    elif mean_b > 1e-10:
        pct = float('inf')
    else:
        pct = 0.0
    # Check from-zero pattern
    from_zero = (mean_a < 1e-10 and mean_b > 1e-10)
    n_consistent = sum(1 for va in vals_a for vb in vals_b
                       if (mean_b > mean_a and vb > va) or (mean_b <= mean_a and vb < va))
    return {
        'p': round(p_val, 4), 'pct': round(pct, 1),
        'dir': '+' if mean_b > mean_a else '-',
        'sig': '*' if p_val < 0.05 else ('(t)' if p_val < 0.10 else ''),
        'cons': f'{n_consistent}/{n_a * n_b}',
        'mean_a': round(mean_a, 4), 'mean_b': round(mean_b, 4),
        'n_a': n_a, 'n_b': n_b, 'from_zero': from_zero,
    }

def fmt(res):
    if res is None:
        return 'ND'
    if res['from_zero']:
        return f'{res["sig"]}from_zero p={res["p"]:.3f}'
    return f'{res["sig"]}{res["dir"]}{abs(res["pct"]):.0f}% p={res["p"]:.3f}'

print('Functions defined')

Functions defined


In [7]:
# Cell 4: TABLE 5 GENES — Full trajectory (NL→IT, NL→IA, NL→CR, IT→CR, IA→CR)
# Classify: IT-persistent, IA-persistent, CR-unique

table5_combos = [
    # Table 5 from-zero genes
    ('MEFV', 'CD8_T', 'Liver'),
    ('MEFV', 'B', 'Blood'),
    ('MEFV', 'NK', 'Blood'),
    ('COL1A1', 'CD8_T', 'Blood'),
    ('FN1', 'CD4_T', 'Blood'),
    ('NLRP3', 'B', 'Blood'),
    # Table 5 quantitative
    ('DNMT3A', 'Myeloid', 'Liver'),
    ('GNLY', 'PlasmaB', 'Liver'),
]

comparisons = [('NL','IT'), ('NL','IA'), ('NL','CR'), ('IT','CR'), ('IA','CR')]

print('='*140)
print('TABLE 5 CR-SPECIFIC GENES: Full Disease Trajectory')
print('='*140)
print(f'{"Gene":>8s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL->IT":>18s} | {"NL->IA":>18s} | {"NL->CR":>18s} | '
      f'{"IT->CR":>18s} | {"IA->CR":>18s} | {"Pattern":>15s}')
print('-'*140)

for gene, lin, tis in table5_combos:
    results = {}
    for ga, gb in comparisons:
        results[f'{ga}->{gb}'] = fast_donor_test(gene, lin, tis, ga, gb)

    # Classify
    nl_it = results.get('NL->IT')
    nl_ia = results.get('NL->IA')
    nl_cr = results.get('NL->CR')

    nl_it_sig = nl_it and nl_it['p'] < 0.05
    nl_ia_sig = nl_ia and nl_ia['p'] < 0.05
    nl_cr_sig = nl_cr and nl_cr['p'] < 0.05

    if nl_it_sig and nl_cr_sig:
        pattern = 'IT-persistent'
    elif not nl_it_sig and nl_ia_sig and nl_cr_sig:
        pattern = 'IA-persistent'
    elif not nl_it_sig and not nl_ia_sig and nl_cr_sig:
        pattern = 'CR-unique'
    elif nl_cr_sig:
        pattern = 'CR-sig (other)'
    else:
        pattern = 'NS at CR'

    print(f'{gene:>8s} | {lin:>8s} | {tis:>6s} | '
          f'{fmt(results.get("NL->IT")):>18s} | '
          f'{fmt(results.get("NL->IA")):>18s} | '
          f'{fmt(results.get("NL->CR")):>18s} | '
          f'{fmt(results.get("IT->CR")):>18s} | '
          f'{fmt(results.get("IA->CR")):>18s} | '
          f'{pattern:>15s}')

TABLE 5 CR-SPECIFIC GENES: Full Disease Trajectory
    Gene |  Lineage | Tissue |             NL->IT |             NL->IA |             NL->CR |             IT->CR |             IA->CR |         Pattern
--------------------------------------------------------------------------------------------------------------------------------------------
    MEFV |    CD8_T |  Liver | (t)from_zero p=0.074 |  from_zero p=0.361 | *from_zero p=0.009 |       +27% p=0.237 |  (t)+1288% p=0.057 |       CR-unique
    MEFV |        B |  Blood |  from_zero p=0.424 |  from_zero p=0.371 | *from_zero p=0.017 |      +202% p=0.112 |      +748% p=0.199 |       CR-unique
    MEFV |       NK |  Blood |  from_zero p=0.424 |  from_zero p=0.131 | *from_zero p=0.017 |      +312% p=0.112 |     +1459% p=0.212 |       CR-unique
  COL1A1 |    CD8_T |  Blood |        -0% p=1.000 |  from_zero p=0.371 | *from_zero p=0.017 | *from_zero p=0.017 |       -52% p=0.359 |       CR-unique
     FN1 |    CD4_T |  Blood |  from_zero p=0.

In [9]:
# Cell 5: IT-significant genes from Result 2: persistence to CR?
# This answers: does CR still carry IT-phase suppressive residue?

sixlayer_combos = [
    # Layer 1: Myeloid paracrine
    ('TGFB1', 'Myeloid', 'Blood'),
    ('LGALS9', 'Myeloid', 'Blood'),
    ('AIM2', 'Myeloid', 'Blood'),
    ('MEFV', 'Myeloid', 'Blood'),
    # Layer 2: Epigenetic
    ('DNMT1', 'Myeloid', 'Blood'),
    ('DNMT1', 'Myeloid', 'Liver'),
    ('DNMT3A', 'Myeloid', 'Blood'),
    ('TET2', 'Myeloid', 'Blood'),
    # Layer 3: Metabolic
    ('MTOR', 'Myeloid', 'Blood'),
    ('MTOR', 'Myeloid', 'Liver'),
    # Layer 4: JAK-STAT
    ('SOCS1', 'CD4_T', 'Blood'),
    ('SOCS3', 'CD4_T', 'Blood'),
    ('JAK1', 'Myeloid', 'Blood'),
    # Layer 5: Exhaustion
    ('TOX', 'CD4_T', 'Liver'),
    ('TOX', 'CD8_T', 'Liver'),
    ('LAYN', 'CD4_T', 'Liver'),
    ('TIGIT', 'CD4_T', 'Liver'),
    ('TIGIT', 'CD8_T', 'Liver'),
    ('BCL6', 'CD8_T', 'Liver'),
    # Layer 6: Differentiation block
    ('PRDM1', 'CD4_T', 'Liver'),
    ('PRDM1', 'CD4_T', 'Blood'),
    ('RORC', 'CD4_T', 'Liver'),
    # Additional
    ('FOXP3', 'CD4_T', 'Liver'),
    ('HLA-DPB1', 'Myeloid', 'Liver'),
    ('HLA-DPB1', 'Myeloid', 'Blood'),
]

print('='*150)
print('IT-significant genes from Result 2: IT→CR Persistence Check')
print('='*150)
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL->IT":>18s} | {"NL->IA":>18s} | {"NL->CR":>18s} | '
      f'{"CR Status":>15s}')
print('-'*150)

for gene, lin, tis in sixlayer_combos:
    nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
    nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')
    nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')

    # Classify CR status relative to IT
    it_sig = nl_it and nl_it['p'] < 0.05
    cr_sig = nl_cr and nl_cr['p'] < 0.05
    ia_sig = nl_ia and nl_ia['p'] < 0.05

    if it_sig and cr_sig:
        # Check same direction
        same_dir = nl_it['dir'] == nl_cr['dir']
        status = 'PERSISTS to CR' if same_dir else 'REVERSED at CR'
    elif it_sig and not cr_sig:
        status = 'RESOLVED at CR'
    elif not it_sig and cr_sig:
        status = 'CR-new'
    else:
        status = 'NS both'

    print(f'{gene:>10s} | {lin:>8s} | {tis:>6s} | '
          f'{fmt(nl_it):>18s} | {fmt(nl_ia):>18s} | '
          f'{fmt(nl_cr):>18s} | {status:>15s}')

IT-significant genes from Result 2: IT→CR Persistence Check
      Gene |  Lineage | Tissue |             NL->IT |             NL->IA |             NL->CR |       CR Status
------------------------------------------------------------------------------------------------------------------------------------------------------
     TGFB1 |  Myeloid |  Blood |     *+156% p=0.008 |     *+176% p=0.016 |     *+120% p=0.036 |  PERSISTS to CR
    LGALS9 |  Myeloid |  Blood |      *+94% p=0.016 |      *+96% p=0.032 |      *+71% p=0.036 |  PERSISTS to CR
      AIM2 |  Myeloid |  Blood |    *+1431% p=0.011 |    *+1705% p=0.018 |     *+810% p=0.033 |  PERSISTS to CR
      MEFV |  Myeloid |  Blood |     *+201% p=0.008 |      +146% p=0.191 |     *+165% p=0.036 |  PERSISTS to CR
     DNMT1 |  Myeloid |  Blood |     *+164% p=0.008 |     *+180% p=0.016 |     *+118% p=0.036 |  PERSISTS to CR
     DNMT1 |  Myeloid |  Liver |     *+125% p=0.002 |       +50% p=0.329 |       +66% p=0.167 |  RESOLVED at CR
    D

In [10]:
# Cell 6: BROAD SCAN — All genes, NL→CR significant, with IT/IA context

lineages = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
tissues = ['Liver', 'Blood']

print('='*150)
print('ALL NL->CR SIGNIFICANT CHANGES (with IT/IA trajectory)')
print('='*150)
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL->IT":>18s} | {"NL->IA":>18s} | {"NL->CR":>18s} | '
      f'{"Pattern":>15s}')
print('-'*150)

cr_results = []

for gene in target_genes:
    for lin in lineages:
        for tis in tissues:
            nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')
            if nl_cr is None or nl_cr['p'] >= 0.10:
                continue

            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')

            it_sig = nl_it and nl_it['p'] < 0.05
            ia_sig = nl_ia and nl_ia['p'] < 0.05
            cr_sig = nl_cr['p'] < 0.05

            if it_sig and cr_sig:
                same_dir = nl_it['dir'] == nl_cr['dir']
                pattern = 'IT-persistent' if same_dir else 'IT-reversed'
            elif not it_sig and ia_sig and cr_sig:
                pattern = 'IA-persistent'
            elif not it_sig and not ia_sig and cr_sig:
                pattern = 'CR-unique'
            elif cr_sig:
                pattern = 'CR-sig'
            else:
                pattern = 'CR-trend'

            cr_results.append({
                'Gene': gene, 'Lineage': lin, 'Tissue': tis,
                'NL_IT': fmt(nl_it), 'NL_IA': fmt(nl_ia), 'NL_CR': fmt(nl_cr),
                'NL_IT_p': nl_it['p'] if nl_it else None,
                'NL_IA_p': nl_ia['p'] if nl_ia else None,
                'NL_CR_p': nl_cr['p'],
                'Pattern': pattern,
            })

            print(f'{nl_cr["sig"]:>1s}{gene:>9s} | {lin:>8s} | {tis:>6s} | '
                  f'{fmt(nl_it):>18s} | {fmt(nl_ia):>18s} | '
                  f'{fmt(nl_cr):>18s} | {pattern:>15s}')

print(f'\nTotal NL->CR significant/trend: {len(cr_results)}')

# Pattern summary
df_cr = pd.DataFrame(cr_results)
print(f'\nPattern distribution:')
print(df_cr['Pattern'].value_counts())

ALL NL->CR SIGNIFICANT CHANGES (with IT/IA trajectory)
      Gene |  Lineage | Tissue |             NL->IT |             NL->IA |             NL->CR |         Pattern
------------------------------------------------------------------------------------------------------------------------------------------------------
*     MEFV |  Myeloid |  Blood |     *+201% p=0.008 |      +146% p=0.191 |     *+165% p=0.036 |   IT-persistent
*     MEFV |    CD8_T |  Liver | (t)from_zero p=0.074 |  from_zero p=0.361 | *from_zero p=0.009 |       CR-unique
(t)     MEFV |       NK |  Liver |  from_zero p=0.405 |  from_zero p=0.136 | (t)from_zero p=0.052 |        CR-trend
*     MEFV |       NK |  Blood |  from_zero p=0.424 |  from_zero p=0.131 | *from_zero p=0.017 |       CR-unique
*     MEFV |        B |  Blood |  from_zero p=0.424 |  from_zero p=0.371 | *from_zero p=0.017 |       CR-unique
*   COL1A1 |    CD8_T |  Blood |        -0% p=1.000 |  from_zero p=0.371 | *from_zero p=0.017 |       CR-unique
*   

In [11]:
# Cell 7: Donor-level values for key CR genes

STAGE_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']

key_cr_combos = [
    # From-zero CR-unique
    ('MEFV', 'CD8_T', 'Liver'),
    ('MEFV', 'B', 'Blood'),
    ('MEFV', 'NK', 'Blood'),
    ('COL1A1', 'CD8_T', 'Blood'),
    ('FN1', 'CD4_T', 'Blood'),
    ('NLRP3', 'B', 'Blood'),
    # Quantitative
    ('DNMT3A', 'Myeloid', 'Liver'),
    ('GNLY', 'PlasmaB', 'Liver'),
    # IT-persistent candidates
    ('DNMT1', 'Myeloid', 'Blood'),
    ('DNMT1', 'Myeloid', 'Liver'),
    ('TIGIT', 'CD4_T', 'Liver'),
    ('JAK1', 'Myeloid', 'Blood'),
    ('HLA-DPB1', 'Myeloid', 'Liver'),
]

for gene, lin, tis in key_cr_combos:
    if gene not in gene_expr:
        continue
    mask = (obs[COL_LINEAGE] == lin) & (obs[COL_TISSUE] == tis)
    cells = obs[mask].copy()
    cell_indices = np.where(mask.values)[0]
    cells['expr'] = gene_expr[gene][cell_indices]
    dm = cells.groupby([COL_DONOR, COL_STAGE])['expr'].mean().reset_index()

    means = {}
    for stage in STAGE_ORDER:
        vals = dm[dm[COL_STAGE] == stage]['expr'].dropna().values
        if len(vals) > 0:
            means[stage] = (len(vals), np.mean(vals))

    vals_str = ' | '.join(f'{s}(n={means[s][0]})={means[s][1]:.4f}'
                          for s in STAGE_ORDER if s in means)
    # NL->CR test
    nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')
    cr_str = fmt(nl_cr) if nl_cr else 'ND'

    print(f'{gene:>10s} {lin:>8s} {tis:>6s}: {vals_str} | NL->CR: {cr_str}')

      MEFV    CD8_T  Liver: NL(n=6)=0.0000 | IT(n=6)=0.0013 | IA(n=5)=0.0001 | AR(n=3)=0.0000 | CR(n=3)=0.0016 | NL->CR: *from_zero p=0.009
      MEFV        B  Blood: NL(n=5)=0.0000 | IT(n=5)=0.0017 | IA(n=4)=0.0006 | AR(n=3)=0.0000 | CR(n=3)=0.0050 | NL->CR: *from_zero p=0.017
      MEFV       NK  Blood: NL(n=5)=0.0000 | IT(n=5)=0.0027 | IA(n=4)=0.0007 | AR(n=3)=0.0005 | CR(n=3)=0.0111 | NL->CR: *from_zero p=0.017
    COL1A1    CD8_T  Blood: NL(n=5)=0.0000 | IT(n=5)=0.0000 | IA(n=4)=0.0026 | AR(n=3)=0.0000 | CR(n=3)=0.0012 | NL->CR: *from_zero p=0.017
       FN1    CD4_T  Blood: NL(n=5)=0.0000 | IT(n=5)=0.0001 | IA(n=4)=0.0005 | AR(n=3)=0.0000 | CR(n=3)=0.0005 | NL->CR: *from_zero p=0.017
     NLRP3        B  Blood: NL(n=5)=0.0000 | IT(n=5)=0.0020 | IA(n=4)=0.0007 | AR(n=3)=0.0017 | CR(n=3)=0.0062 | NL->CR: *from_zero p=0.017
    DNMT3A  Myeloid  Liver: NL(n=6)=0.0058 | IT(n=6)=0.0328 | IA(n=5)=0.0261 | AR(n=3)=0.0148 | CR(n=3)=0.0299 | NL->CR: *+412% p=0.022
      GNLY  PlasmaB  Liv

In [12]:
# Cell 8: Save supplementary table

# Build comprehensive CR table
supp_rows = []
for gene in target_genes:
    for lin in lineages:
        for tis in tissues:
            nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')
            if nl_cr is None:
                continue
            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')

            any_sig = (nl_cr['p'] < 0.05 or
                      (nl_it and nl_it['p'] < 0.05) or
                      (nl_ia and nl_ia['p'] < 0.05))
            if not any_sig:
                continue

            it_sig = nl_it and nl_it['p'] < 0.05
            ia_sig = nl_ia and nl_ia['p'] < 0.05
            cr_sig = nl_cr['p'] < 0.05

            if it_sig and cr_sig:
                same_dir = nl_it['dir'] == nl_cr['dir']
                pattern = 'IT-persistent' if same_dir else 'IT-reversed'
            elif not it_sig and ia_sig and cr_sig:
                pattern = 'IA-persistent'
            elif not it_sig and not ia_sig and cr_sig:
                pattern = 'CR-unique'
            elif it_sig and not cr_sig:
                pattern = 'Resolved at CR'
            elif cr_sig:
                pattern = 'CR-sig'
            else:
                pattern = 'Other'

            supp_rows.append({
                'Gene': gene, 'Lineage': lin, 'Tissue': tis,
                'NL->IT': fmt(nl_it),
                'NL->IA': fmt(nl_ia),
                'NL->CR': fmt(nl_cr),
                'Pattern': pattern,
            })

df_supp = pd.DataFrame(supp_rows)
supp_path = os.path.join(SAVE_DIR, 'SuppTable_CR_Trajectory.csv')
df_supp.to_csv(supp_path, index=False, encoding='utf-8-sig')
print(f'Saved: {supp_path}')
print(f'Rows: {len(df_supp)}')
print(f'\nPattern distribution:')
print(df_supp['Pattern'].value_counts())
print(f'\n--- IT-persistent (IT suppression continues to CR) ---')
for _, r in df_supp[df_supp['Pattern'] == 'IT-persistent'].iterrows():
    print(f'  {r["Gene"]:>10s} {r["Lineage"]:>8s} {r["Tissue"]:>6s}: IT={r["NL->IT"]:>18s} CR={r["NL->CR"]:>18s}')
print(f'\n--- CR-unique (absent in IT and IA) ---')
for _, r in df_supp[df_supp['Pattern'] == 'CR-unique'].iterrows():
    print(f'  {r["Gene"]:>10s} {r["Lineage"]:>8s} {r["Tissue"]:>6s}: IT={r["NL->IT"]:>18s} CR={r["NL->CR"]:>18s}')
print(f'\n--- Resolved at CR (IT changes no longer present) ---')
for _, r in df_supp[df_supp['Pattern'] == 'Resolved at CR'].iterrows():
    print(f'  {r["Gene"]:>10s} {r["Lineage"]:>8s} {r["Tissue"]:>6s}: IT={r["NL->IT"]:>18s} CR={r["NL->CR"]:>18s}')

Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/Result4-3-supplementary_26Mar13/SuppTable_CR_Trajectory.csv
Rows: 116

Pattern distribution:
Pattern
IT-persistent     33
Resolved at CR    29
CR-unique         27
Other             14
IA-persistent     13
Name: count, dtype: int64

--- IT-persistent (IT suppression continues to CR) ---
        MEFV  Myeloid  Blood: IT=    *+201% p=0.008 CR=    *+165% p=0.036
      DNMT3A  Myeloid  Blood: IT=    *+348% p=0.012 CR=    *+350% p=0.036
      DNMT3A    CD4_T  Blood: IT=    *+133% p=0.008 CR=    *+114% p=0.036
      DNMT3A  PlasmaB  Liver: IT=    *+749% p=0.012 CR=    *+328% p=0.043
       TGFB1  Myeloid  Blood: IT=    *+156% p=0.008 CR=    *+120% p=0.036
      LGALS9  Myeloid  Blood: IT=     *+94% p=0.016 CR=     *+71% p=0.036
       DNMT1  Myeloid  Blood: IT=    *+164% p=0.008 CR=    *+118% p=0.036
        TET2  Myeloid  Blood: IT=    *+150% p=0.008 CR=    *+138% p=0.036
        MTOR  Myeloid  Liver: IT=    *+327% p=0.034 CR

In [13]:
# Cell 9: Summary

print('='*70)
print('MANUSCRIPT REVISION GUIDE — Result 4.3')
print('='*70)
print()
print('KEY QUESTIONS ANSWERED:')
print('  1. Which IT-phase changes persist to CR?')
print('     → IT-persistent genes: [CHECK Cell 8 output]')
print('  2. Which are IA-phase changes persisting to CR?')
print('     → IA-persistent genes: [CHECK Cell 8 output]')
print('  3. Which are CR-unique (absent in IT/IA)?')
print('     → CR-unique genes: [CHECK Cell 8 output]')
print('     → From-zero genes (MEFV, COL1A1, FN1, NLRP3)')
print('  4. Which IT changes RESOLVE at CR?')
print('     → Resolved genes: important for understanding what heals')
print()
print('NARRATIVE STRUCTURE:')
print('  Para 1: IT-persistent changes (suppressive residue)')
print('  Para 2: CR-unique changes (from-zero lineage-aberrant)')
print('  Para 3: Resolved changes (what normalizes at CR)')
print('  Para 4: Clinical implications (immunologic imprint)')
print()
print('All data saved to:', SAVE_DIR)

MANUSCRIPT REVISION GUIDE — Result 4.3

KEY QUESTIONS ANSWERED:
  1. Which IT-phase changes persist to CR?
     → IT-persistent genes: [CHECK Cell 8 output]
  2. Which are IA-phase changes persisting to CR?
     → IA-persistent genes: [CHECK Cell 8 output]
  3. Which are CR-unique (absent in IT/IA)?
     → CR-unique genes: [CHECK Cell 8 output]
     → From-zero genes (MEFV, COL1A1, FN1, NLRP3)
  4. Which IT changes RESOLVE at CR?
     → Resolved genes: important for understanding what heals

NARRATIVE STRUCTURE:
  Para 1: IT-persistent changes (suppressive residue)
  Para 2: CR-unique changes (from-zero lineage-aberrant)
  Para 3: Resolved changes (what normalizes at CR)
  Para 4: Clinical implications (immunologic imprint)

All data saved to: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/Result4-3-supplementary_26Mar13
